# Data Preparation: Leakage-Free Bidirectional Splitting

## Overview

This notebook implements the data preprocessing pipeline required to fine-tune the NLLB-200 model. To address the scarcity of training data (Low-Resource setting), we employ a Bidirectional Augmentation strategy, training the model to translate both Odia $\rightarrow$ German and German $\rightarrow$ Odia simultaneously.

Crucially, this script enforces a Strict Anti-Leakage Splitting Strategy . The dataset is split into Train, Validation, and Test sets based on unique sentence pairs before augmentation. This ensures that a specific sentence pair (A, B) never appears in the training set as (A $\rightarrow$ B) while appearing in the test set as (B $\rightarrow$ A), which would constitute data leakage and inflate performance metrics.

## Key Steps
1. **Ingestion:** Loads the cleaned corpus of 3,676 unique parallel pairs.
2. **Strategic Splitting:** Divides unique pairs into Train (80%), Validation (10%), and Test (10%) sets prior to any duplication.
3. **Bidirectional Augmentation:** For every pair in each split, generates two training instances with specific task prefixes:
  * translate Odia to German: [Odia Sentence]
  * translate German to Odia: [German Sentence]
4. **Serialization:** Saves the processed, shuffled datasets into separate JSONL files (train_bidirectional.jsonl, etc.) ready for the Hugging Face Trainer.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# import required libraries
import json
import random
import os
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
# --- CONFIGURATION ---
INPUT_FILE = "/content/drive/MyDrive/Research_Paper_Publication/data/transformed/authentic_corpus_final.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/Research_Paper_Publication/data/transformed/"

# Define Output Files for each split
TRAIN_FILE = os.path.join(OUTPUT_DIR, "train_bidirectional.jsonl")
VAL_FILE = os.path.join(OUTPUT_DIR, "val_bidirectional.jsonl")
TEST_FILE = os.path.join(OUTPUT_DIR, "test_bidirectional.jsonl")

# Define the exact prefixes
PREFIX_ORI_TO_DEU = "translate Odia to German: "
PREFIX_DEU_TO_ORI = "translate German to Odia: "

SOURCE_SENTENCE_FIELD = "sentence_ory_Orya"
TARGET_SENTENCE_FIELD = "sentence_deu_Latn"

In [ ]:
def create_bidirectional_instances(records, start_id=1):
    """
    Transforms a list of unique translation pairs into a bidirectional dataset
    suitable for training a sequence-to-sequence model.

    For every single record containing an Odia and a German sentence, this function
    generates two distinct training instances:
    1. Odia -> German (using the defined source-to-target prefix).
    2. German -> Odia (using the defined target-to-source prefix).

    Args:
        records (list of dict): A list of dictionaries, where each dictionary represents
            a single data entry containing metadata and the parallel sentences.
            Expected keys include 'id', 'URL', 'domain', 'topic', 'publication_date',
            and the dynamic keys for source/target sentences.
        start_id (int, optional): The starting integer for the new unique identifier
            assigned to each generated instance. Defaults to 1.

    Returns:
        list of dict: A list of dictionaries representing the bidirectional instances.
            Each dictionary contains:
            - 'id': A new unique sequential ID.
            - 'original_id': The ID from the source record.
            - 'input_text': The source text with the appropriate task prefix.
            - 'target_text': The target translation.
            - Metadata fields (URL, domain, etc.).
    """
    bidirectional_data = []
    current_id = start_id

    for record in records:
        # Extract metadata and content fields safely using .get()
        original_id = record.get('id')
        url = record.get('URL')
        domain = record.get('domain')
        topic = record.get('topic')
        pub_date = record.get('publication_date')

        # Extract the actual parallel sentences using global field constants
        odia_sentence = record.get(SOURCE_SENTENCE_FIELD)
        german_sentence = record.get(TARGET_SENTENCE_FIELD)

        # validation: Ensure both sentences exist and are strings before processing.
        # This skips incomplete or malformed records.
        if not isinstance(odia_sentence, str) or not isinstance(german_sentence, str):
            continue

        # -------------------------------------------------------
        # Instance 1: Forward Translation (Odia -> German)
        # -------------------------------------------------------
        # Prepend the specific prefix token (e.g., 'translate Odia to German: ')
        bidirectional_data.append({
            "id": current_id,
            "original_id": original_id,
            "URL": url,
            "domain": domain,
            "topic": topic,
            "publication_date": pub_date,
            "input_text": PREFIX_ORI_TO_DEU + odia_sentence,
            "target_text": german_sentence
        })
        current_id += 1

        # -------------------------------------------------------
        # Instance 2: Reverse Translation (German -> Odia)
        # -------------------------------------------------------
        # Swap source/target and apply the reverse prefix token
        bidirectional_data.append({
            "id": current_id,
            "original_id": original_id,
            "URL": url,
            "domain": domain,
            "topic": topic,
            "publication_date": pub_date,
            "input_text": PREFIX_DEU_TO_ORI + german_sentence,
            "target_text": odia_sentence
        })
        current_id += 1

    return bidirectional_data

In [ ]:
def process_data():
    """
    Orchestrates the end-to-end data preparation pipeline for training.

    This function executes a leakage-free data split strategy by:
    1. Loading the original unique translation pairs.
    2. Splitting these unique pairs *before* augmentation into Train (80%),
       Validation (10%), and Test (10%) sets. This ensures that a specific
       sentence pair never exists in both the training and test sets (e.g.,
       A->B in Train and B->A in Test).
    3. Generating bidirectional instances (Odia->German and German->Odia)
       independently for each split.
    4. Saving the final processed datasets to separate JSONL files.

    Global Constants Required:
        INPUT_FILE (str): Path to the source JSONL containing unique pairs.
        TRAIN_FILE (str): Output path for the training set.
        VAL_FILE (str): Output path for the validation set.
        TEST_FILE (str): Output path for the test set.
    """
    # 1. Validation: Ensure the source file exists before proceeding.
    if not os.path.exists(INPUT_FILE):
        print(f"⛔️ ERROR: Input file '{INPUT_FILE}' not found.")
        return

    print(f"Reading original structured corpus from '{INPUT_FILE}'...")

    # Load the dataset line-by-line to handle potentially large JSONL files efficiently.
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        original_data = [json.loads(line) for line in f]

    print(f"Total unique pairs found: {len(original_data)}")

    # --- CRITICAL STEP: SPLIT UNIQUE PAIRS FIRST ---
    # We split the unique entries (original_data) BEFORE creating bidirectional copies.
    # This prevents "Data Leakage". If we augmented first and then split,
    # the model could memorize "A -> B" in train and be tested on "B -> A",
    # effectively cheating.

    # Target Split Ratios: 80% Train, 10% Validation, 10% Test

    # Step 1: Isolate the Test set (10% of total)
    train_val_subset, test_subset = train_test_split(
        original_data, test_size=0.10, random_state=42, shuffle=True
    )

    # Step 2: Split the remaining 90% into Train and Validation.
    # To get 10% of the ORIGINAL total for validation, we need 1/9th of the
    # remaining data. (0.10 / 0.90 ≈ 0.1111)
    train_subset, val_subset = train_test_split(
        train_val_subset, test_size=0.1111, random_state=42, shuffle=True
    )

    print(f"\n--- Split Statistics (Unique Pairs) ---")
    print(f"Train pairs: {len(train_subset)}")
    print(f"Val pairs:   {len(val_subset)}")
    print(f"Test pairs:  {len(test_subset)}")

    # --- AUGMENTATION ---
    # Now that the unique pairs are safely separated, we generate the
    # bidirectional copies (Source->Target and Target->Source) for each split.
    # This doubles the dataset size.
    print("\nGenerating bidirectional instances...")

    # We pass sequential start_ids to ensure every record across all files has a unique ID.
    final_train = create_bidirectional_instances(train_subset, start_id=1)
    final_val = create_bidirectional_instances(val_subset, start_id=len(final_train)+1)
    final_test = create_bidirectional_instances(test_subset, start_id=len(final_train)+len(final_val)+1)

    # --- SHUFFLE ---
    # Shuffle within the respective splits to ensure the model doesn't see
    # "Odia->German" immediately followed by "German->Odia" during training batches.
    random.shuffle(final_train)
    random.shuffle(final_val)
    random.shuffle(final_test)

    # --- SAVE TO SEPARATE FILES ---
    # Helper function to encapsulate the JSONL writing logic
    def save_jsonl(data, filename):
        with open(filename, 'w', encoding='utf-8') as f_out:
            for instance in data:
                f_out.write(json.dumps(instance, ensure_ascii=False) + '\n')
        print(f"Saved {len(data)} instances to {filename}")

    save_jsonl(final_train, TRAIN_FILE)
    save_jsonl(final_val, VAL_FILE)
    save_jsonl(final_test, TEST_FILE)

    print("\n✅ Leakage-free split complete! You now have 3 separate files.")

if __name__ == "__main__":
    process_data()

Reading original structured corpus from '/content/drive/MyDrive/Research_Paper_Publication/data/transformed/authentic_corpus_final.jsonl'...
Total unique pairs found: 3676

--- Split Statistics (Unique Pairs) ---
Train pairs: 2940
Val pairs:   368
Test pairs:  368

Generating bidirectional instances...
Saved 5880 instances to /content/drive/MyDrive/Research_Paper_Publication/data/transformed/train_bidirectional.jsonl
Saved 736 instances to /content/drive/MyDrive/Research_Paper_Publication/data/transformed/val_bidirectional.jsonl
Saved 736 instances to /content/drive/MyDrive/Research_Paper_Publication/data/transformed/test_bidirectional.jsonl

✅ Leakage-free split complete! You now have 3 separate files.
